# 3 Predictive Analytics with a) Support Vector Machines and b) Neural Networks

Following the descriptive analysis in Section 2, this section constructs two supervised models that predict taxi trip demand per spatial unit and time bucket in spatio-temporal resolution. The objective is not to forecast bucket x+1 from bucket x, but to estimate demand conditional on calendar and environmental context, for example, *"what demand can be expected on a rainy Sunday from 10-11 am around the outskirts of the city?"* Features are therefore restricted to calendar, weather, and spatial identity; the lagged demand of the immediately preceding bucket is deliberately excluded.

The canonical modeling dataset is the **H3 r6 x 4h aggregation** with cells zero-filled so that the absence of trips is represented as an observation rather than a missing row.

## Structure Rationale

The section is organised as a shared modeling foundation followed by the two model families and a comparative synthesis. Feature design and the validation strategy (3.1-3.2) are fixed first, since the resulting feature matrix, data splits, and evaluation metrics are reused by every subsequent subsection to keep results comparable. The two model families are then developed independently against this common interface. Support vector machines in 3.3-3.4 and a feedforward neural network in 3.5-3.6. Resolution sensitivity (3.7) and the holdout comparison (3.8) consume the trained models and are therefore placed last.

The feature matrix, data splits, and the evaluation metrics (MAE, RMSE, R2) are defined once in 3.1-3.2 and reused throughout, so that all reported numbers are comparable across models and resolutions.

### 3.1 Modeling dataset and feature design

The aggregated panel is reframed as a supervised regression problem with target `demand_count` and feature matrix `X = [calendar, weather, spatial identity]`. Required outputs:

- Definition of the target `demand_count` and confirmation of the zero-filled structure (every cell x bucket present).
- Calendar features derived from `timestamp`: time-of-day bucket, day-of-week, weekend indicator, month/season. Cyclical (sin/cos) encodings are applied where continuity matters (23:00 -> 00:00).
- Weather features as already present in the dataset (`temp_4h`, `rain_4h`, `snow_4h`, `precip_4h`, `wind_speed_4h`, `weather_code_4h`).
- A spatial encoding of each H3 cell. The choice between cell-centroid latitude/longitude (continuous, permitting spatial interpolation) and one-hot encoding (high cardinality) is documented and justified.
- A clean feature matrix and target reused by all downstream subsections. To avoid leakage, lagged or same-bucket demand of the target is not included.

### 3.2 Validation strategy

A single validation procedure is defined and reused across both model families to ensure comparability. Required outputs:

- Justification of the split. Because the prediction target is demand under a given condition, a time-based holdout (training on earlier months, testing on the most recent block) most closely reflects deployment and limits leakage of near-duplicate buckets across sets. A random split may additionally be reported for contrast.
- An explicit statement of how the spatial dimension is treated: all cells appear in both training and test sets, since prediction is for known cells under unseen conditions.
- A fixed train/validation/test partition and a common metric set (MAE, RMSE, R2) used by every later subsection.
- Hyperparameter search (3.4) and network tuning (3.6) draw only on the training and validation folds; the holdout is reserved for final reporting.

### 3.3 SVM - linear baseline (no kernel)

Following the assignment, the SVM is first specified without a kernel. Required outputs:

- Feature standardisation (SVMs are scale-sensitive), with the scaler fitted on the training partition only.
- A linear SVM regressor trained on the training split defined in 3.2.
- MAE, RMSE, and R2 on the holdout using the common metric set.
- A brief interpretation of which features carry weight and of the practical meaning of the error (e.g. average deviation in trips per cell per 4h bucket).

### 3.4 SVM - kernels and grid search

Model complexity is increased through non-linear kernels with tuned hyperparameters. Required outputs:

- RBF and, optionally, polynomial kernels.
- A grid search over the relevant hyperparameters (`C`, `gamma`, `epsilon`) on the training and validation folds only. As SVR scales poorly with sample size, a justified subsample or representative training slice may be used.
- Best hyperparameters and holdout performance reported against the linear baseline.
- An evaluation of model quality and a discussion of its shortfalls.

### 3.5 Neural network - feedforward baseline

The procedure of the SVM subsections is repeated with a feedforward neural network. Required outputs:

- Reuse of the same feature matrix, splits, scaling, and metrics from 3.1-3.2 so that the later comparison is fair.
- A feedforward multilayer perceptron: an input layer sized to the feature count, several hidden layers with non-linear activations, and a single linear output for the demand regression.
- Training with early stopping on the validation fold, and MAE, RMSE, and R2 on the holdout.
- A statement of the loss, optimiser, and stopping criterion and the reasons for their selection.

### 3.6 Neural network - tuning

Required outputs:

- Exploration of architecture and regularisation: number and width of layers, dropout, learning rate, batch size, and epochs.
- All tuning confined to the training and validation folds, with a single final holdout evaluation.
- The resulting holdout metrics recorded for the comparison in 3.8, alongside a note on whether performance differs materially from the SVM.

### 3.7 Resolution sensitivity

Model performance is examined as spatial and temporal resolution is varied. Required outputs:

- The best SVM and best neural network re-evaluated at different spatial resolutions (e.g. H3 r5 coarser, r7 finer) and at census-tract / community-area units.
- The same models re-evaluated at different temporal resolutions (e.g. 1h, 4h, daily) using the aggregation produced in Section 1.
- A table of MAE, RMSE, and R2 across the resolution grid for both models, with a visualisation of the granularity-sparsity trade-off.
- An interpretation of where each model degrades and which resolution best serves the operational question.

### 3.8 Model comparison, shortfalls, and improvement levers

The two model families are compared on a common holdout at identical spatial and temporal resolution. Required outputs:

- A side-by-side holdout table (SVM versus neural network) reporting MAE, RMSE, and R2 together with training and inference cost.
- An explicit assessment of whether a deep-learning approach is warranted for this dataset and the client context.
- Improvement levers for a follow-up project (e.g. point-of-interest features, richer weather and temporal features, gradient boosting, spatial embeddings, additional data).
- A non-technical interpretation translating the chosen model's error into an implication for fleet operations.

### Summary and hand-off

A concise recap of the chosen feature set and validation strategy (3.1-3.2), the best SVM (3.3-3.4), the best neural network (3.5-3.6), the effect of resolution on both (3.7), and the resulting recommendation (3.8). These results inform the Discussion and Outlook in Section 5.